# Colab C — workers без discover

Runbook: `Fetcher/docs/COLAB_20K_RUN_RU.md` · Аудит: `Fetcher/docs/DATASET_20K_AUDIT_20k-test-3_RU.md`.

**HF_TOKEN** — только Colab Secret `HF_TOKEN`, не в коде.

**Преднастроен для Colab C, поставь `worker_shard_index=2`, `output_dir=.../20k-worker-c`, `worker_id=colab-c`.

## 0. CONFIG

In [ ]:
# === отредактируй ===
CONFIG = {
    "repo_url": "https://github.com/lebedev-ilia/TrendFlowML.git",
    "fetcher": "/content/TrendFlowML/Fetcher",
    "drive_secrets": "/content/drive/MyDrive/trendflow_secrets",
    "hf_repo_prefix": "Ilialebedev",
    "interval_sec": 120,
    "metrics_workers": 9096,
    "metrics_discover": 9095,
}
CONFIG["output_dir"] = "/content/drive/MyDrive/dataset_runs/20k-worker-c"
CONFIG["worker_id"] = "colab-c"
CONFIG["worker_shard_index"] = 2
CONFIG["worker_shard_count"] = 3

import os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output_dir:", OUTPUT_DIR)


## 1. Drive + git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

dest = Path("/content/TrendFlowML")
if (dest / ".git").exists():
    subprocess.run(["git", "-C", str(dest), "pull"], check=False)
    print("git pull")
else:
    subprocess.run(["git", "clone", CONFIG["repo_url"], str(dest)], check=True)
    print("git clone")


## 2. Зависимости

In [ ]:
%cd /content/TrendFlowML/Fetcher
!apt-get update -qq && apt-get install -y -qq nodejs
!python -m pip install -q -U pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q -U huggingface_hub yt-dlp "pytubefix>=9.5.0" nodejs-wheel-binaries google-api-python-client


## 3. Cookies, keys, HF_TOKEN

In [ ]:
import os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
secrets = Path(CONFIG["drive_secrets"])
cookies = FETCHER / "fetcher/dataset_collector/cookies"
keys = FETCHER / "fetcher/dataset_collector/keys"
cookies.mkdir(parents=True, exist_ok=True)
keys.mkdir(parents=True, exist_ok=True)

import shutil
for p in (secrets / "cookies").glob("*.txt"):
    shutil.copy2(p, cookies / p.name)
shutil.copy2(secrets / "keys.txt", keys / "keys.txt")

try:
    from google.colab import userdata
    t = (userdata.get("HF_TOKEN") or "").strip()
    if t:
        os.environ["HF_TOKEN"] = t
except Exception:
    pass
if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError("Set Colab Secret HF_TOKEN")

kf = keys / "keys.txt"
os.environ["FETCHER_YOUTUBE_DATA_API_KEYS"] = kf.read_text().strip().replace("\n", ",")
print("API keys:", len(os.environ["FETCHER_YOUTUBE_DATA_API_KEYS"].split(",")))


## 4. Drive token (subprocess)

In [ ]:
%cd /content/TrendFlowML/Fetcher
!python scripts/export_colab_drive_token.py --output-dir "{CONFIG["output_dir"]}"


## 5. Workers (без discover)

Без `upload-hf-shards` — metadata пушит Colab A.

In [ ]:
import subprocess, os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--role", "workers",
    "--output-dir", CONFIG["output_dir"],
    "--hf-repo-prefix", CONFIG["hf_repo_prefix"],
    "--interval", str(CONFIG["interval_sec"]),
    "--metrics-port", str(CONFIG["metrics_workers"]),
    "--worker-id", CONFIG["worker_id"],
    "--worker-shard-index", str(CONFIG["worker_shard_index"]),
    "--worker-shard-count", str(CONFIG["worker_shard_count"]),
    "--hf-coord",
    "--worker-kinds", "download,enrich-metadata,upload-hf-videos,upload-hf-enrich",
]

log = Path("/content/workers_log.txt")
with open(log, "w") as fh:
    p = subprocess.Popen(cmd, cwd=str(FETCHER), stdout=fh, stderr=subprocess.STDOUT, env=os.environ.copy())
print("workers pid", p.pid)


## 6. Мониторинг

In [ ]:
%cd /content/TrendFlowML/Fetcher
!bash scripts/colab_monitoring_native.sh install
!bash scripts/colab_monitoring_native.sh start
!curl -sf http://127.0.0.1:9096/metrics | grep dataset_collector_coord | head -5


## 7. Логи

In [ ]:
!tail -40 /content/workers_log.txt
!python scripts/colab_20k_bootstrap.py --role status --output-dir "{CONFIG["output_dir"]}"


## 8. Stop

In [ ]:
!pkill -f fetcher.dataset_collector || true
!pkill -f colab_20k_bootstrap || true
